## Enriching Great British Bake Off's Technical Challenge Information

This notebook enriches publically available GBBO episode information with the Technical Challenge Score found on https://thegreatbritishbakeoff.co.uk/recipes/all/?collection=Technical+bakes&showFilter=false 

In [ ]:
import os
import sys
import random
import time
import requests
import re

from playwright.async_api import async_playwright, expect
from fuzzywuzzy import fuzz

In [ ]:
!pip install playwright

In [ ]:
!which playwright

In [ ]:
!playwright install

In [ ]:
# Start the browser
playwright = await async_playwright().start()

In [ ]:
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()
# await browser.close()

In [ ]:
async def open_browser(headless=False):
    """
    Starts the automated browser and opens a new window
    """
    # Start playwright
    playwright = await async_playwright().start()

    # Open chromium (chrome) browser, can use firefox or others
    browser = await playwright.chromium.launch(headless=headless)
  
    # Create a new browser window
    page = await browser.new_page()

    return browser, page

In [ ]:
# visit a URL
url = 'https://thegreatbritishbakeoff.co.uk/recipes/all/?collection=Technical+bakes&showFilter=false'
await page.goto(url)

In [ ]:
recipe_info = page.locator('h5').all_inner_texts()
await recipe_info

In [ ]:
challenge_locator = page.locator('span.difficulty-level')
await challenge_locator.first.wait_for(state='visible')

scores = await challenge_locator.all()
scores_list = []

for score in scores:
    challenge_score = await score.get_attribute('aria-label')
    scores_list.append(challenge_score)
print(scores_list)

For each page I need to:
- Navigate to the page
- get the recipe_title
- get the difficulty level

In [ ]:
technical_challenge_info = []

while True:
    pagination_locator = page.locator('div.recipes-loop__pagination')
    await pagination_locator.first.wait_for(state="visible")
    
    # 1. Target the individual recipe cards on the current page grid
    # (Using the standard GBBO recipe layout class)
    recipe_cards = await page.locator('.recipes-loop__grid .recipes-loop__item').all()
    
    # Track if we found any scores on this specific page
    page_had_scores = False
    
    for card in recipe_cards:
        score_locator = card.locator('span.difficulty-level')
        
        # 2. ONLY process this card if a difficulty score exists inside it
        if await score_locator.count() > 0:
            page_had_scores = True
            
            # Fetch the matching title inside this specific card
            title_text = await card.locator('h5').inner_text()
            
            # Fetch and clean the score string
            score_text = await score_locator.get_attribute('aria-label')
            challenge_level = score_text.replace("Difficulty level: ", "").strip() if score_text else None
            
            technical_challenge_info.append({
                'title': title_text,
                'challenge level': challenge_level
            })
            
    # 3. Optimization: If an entire page had ZERO scores, you've hit the point 
    # where they stop. You can break early and save execution time.
    if not page_had_scores:
        print("Reached pages without challenge scores. Stopping scraping.")
        break
        
    # 4. Handle standard pagination click
    next_page_locator = page.locator('.recipes-loop__pagination .page-numbers.current + a')
    if await next_page_locator.count() > 0:
        await next_page_locator.click(force=True)
        await page.wait_for_load_state("networkidle")
    else:
        break

print(f"Scraped {len(technical_challenge_info)} recipes with difficulty scores.")
print(technical_challenge_info)


In [61]:
import pandas as pd
import numpy as np

challenge_score_info_df = pd.json_normalize(technical_challenge_info)
challenge_score_info_df

,title,challenge level
0,CHERISH FINDEN’S VEGAN FRUIT TARTS,Needs skill
1,PAUL HOLLYWOOD’S STEAMED SYRUP PUDDINGS WITH C...,Needs skill
2,CHERISH FINDEN’S TARTE TATIN,Needs skill
3,CHERISH FINDEN’S MATCHA AND YUZU SANDWICH BISC...,Easy
4,RAVNEET GILL’S STRAWBERRIES AND CLOTTED CREAM ...,Challenging
...,...,...
229,PRUE LEITH’S STROOPWAFELS,Needs skill
230,PAUL HOLLYWOOD’S CLASSIC COTTAGE LOAF,Easy
231,PAUL HOLLYWOOD’S FORTUNE COOKIES,Needs skill
232,PRUE LEITH’S CHOCOLATE MINI ROLLS,Needs skill


In [62]:
#Challenge score mapping
challenge_score_mapping = {
    'Challenging' : 3,
    'Needs skill' : 2,
    'Easy' : 1
}

challenge_score_info_df['challenge level numeric'] = challenge_score_info_df['challenge level'].map(challenge_score_mapping)
challenge_score_info_df

,title,challenge level,challenge level numeric
0,CHERISH FINDEN’S VEGAN FRUIT TARTS,Needs skill,2
1,PAUL HOLLYWOOD’S STEAMED SYRUP PUDDINGS WITH C...,Needs skill,2
2,CHERISH FINDEN’S TARTE TATIN,Needs skill,2
3,CHERISH FINDEN’S MATCHA AND YUZU SANDWICH BISC...,Easy,1
4,RAVNEET GILL’S STRAWBERRIES AND CLOTTED CREAM ...,Challenging,3
...,...,...,...
229,PRUE LEITH’S STROOPWAFELS,Needs skill,2
230,PAUL HOLLYWOOD’S CLASSIC COTTAGE LOAF,Easy,1
231,PAUL HOLLYWOOD’S FORTUNE COOKIES,Needs skill,2
232,PRUE LEITH’S CHOCOLATE MINI ROLLS,Needs skill,2


In [63]:
#cleaning title string

challenge_score_info_df['title'] = challenge_score_info_df['title'].str.lower()

title_str_to_replace = [
    "mary berry’s",
    "paul hollywood’s",
    "prue leith’s",
    "cherish finden’s",
    "ravneet gill’s",
    "liam charles’s",
    "caroline waldegrave’s"
]

challenge_score_info_df['title'] = challenge_score_info_df['title'].replace(title_str_to_replace, "", regex=True)
challenge_score_info_df

,title,challenge level,challenge level numeric
0,vegan fruit tarts,Needs skill,2
1,steamed syrup puddings with custard,Needs skill,2
2,tarte tatin,Needs skill,2
3,matcha and yuzu sandwich biscuits,Easy,1
4,strawberries and clotted cream danish,Challenging,3
...,...,...,...
229,stroopwafels,Needs skill,2
230,classic cottage loaf,Easy,1
231,fortune cookies,Needs skill,2
232,chocolate mini rolls,Needs skill,2


In [64]:
challenge_score_info_df.to_csv('TechnicalChallengeScores.csv')

In [65]:
episode_info_df = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/Episodes.csv')
episode_info_df['Technical'] = episode_info_df['Technical'].str.lower()
episode_info_df['Theme'] = episode_info_df['Theme'].str.lower()
episode_info_df['Signature'] = episode_info_df['Signature'].str.lower()
episode_info_df['Showstopper'] = episode_info_df['Showstopper'].str.lower()
episode_info_df_clean = episode_info_df.drop(columns=['MyRating (out of 10)', 'MyViewership'])
episode_info_df_clean

,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min)
0,8/17/2010,1,1,cake,cake,180.0,victoria sandwich,NaN,chocolate celebration cake,NaN
1,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0
2,8/31/2010,1,3,bread,signature bread,210.0,cob,150.0,12 sweet and 12 savoury rolls,360.0
3,9/7/2010,1,4,pudding,classic pudding,150.0,mini hot lemon soufflés,40.0,"crumble, bread, & suet puddings",300.0
4,9/14/2010,1,5,pastry,savoury pie,150.0,cornish pasties,90.0,savory canapés and sweet tartlets,300.0
...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,12 individual spiced buns,165.0,lemon and thyme drizzle cake,90.0,floral dessert,240.0
130,11/7/2023,14,7,desserts,8 creme caramels,165.0,orange and ginger treacle puddings,90.0,meringue bombe,NaN
131,11/14/2023,14,8,party,12 sausage rolls,150.0,caterpiller cake,150.0,anything but beige buffet,270.0
132,11/21/2023,14,9,patisserie,24 financiers,120.0,tart aux pommes,150.0,millefoglie,240.0


In [66]:
# Source - https://stackoverflow.com/a/77420428
# Posted by tetris programming
# Retrieved 2026-07-22, License - CC BY-SA 4.0

episode_info_df_clean["challenge level numeric"] = np.nan
sim_score = 80

for row_counter in range(episode_info_df_clean.shape[0]):
    searched_technical = episode_info_df_clean.iloc[row_counter]["Technical"]
    if not isinstance(searched_technical, str):
        continue

    # Compute the fuzzy score for every row in challenge_score_info_df
    scores = challenge_score_info_df.apply(
        lambda x: fuzz.partial_ratio(x['title'], searched_technical) if isinstance(x['title'], str) else 0,
        axis=1
    )

    best_score = scores.max()
    if best_score > sim_score:
        best_idx = scores.idxmax()  # position/label of the best-scoring row
        best_value = challenge_score_info_df.loc[best_idx, "challenge level numeric"]

        episode_info_df_clean.iloc[
            row_counter, episode_info_df_clean.columns.get_loc("challenge level numeric")
        ] = best_value

print(episode_info_df_clean)


        Airdate  Season  Episode       Theme                  Signature  \
0     8/17/2010       1        1        cake                       cake   
1     8/24/2010       1        2     biscuit       personality biscuits   
2     8/31/2010       1        3       bread            signature bread   
3      9/7/2010       1        4     pudding            classic pudding   
4     9/14/2010       1        5      pastry                savoury pie   
..          ...     ...      ...         ...                        ...   
129  10/31/2023      14        6   botanical  12 individual spiced buns   
130   11/7/2023      14        7    desserts           8 creme caramels   
131  11/14/2023      14        8       party           12 sausage rolls   
132  11/21/2023      14        9  patisserie              24 financiers   
133  11/28/2023      14       10       final                  8 éclairs   

     Signature Time (min)                           Technical  \
0                   180.0         

In [67]:
# How many rows got a match vs. stayed NaN?
print(episode_info_df_clean["challenge level numeric"].isna().sum(), "unmatched out of", len(episode_info_df_clean))

# Spot-check a few matches to make sure they're sensible, not false positives
print(episode_info_df_clean[["Technical", "challenge level numeric"]].sample(10))

66 unmatched out of 134
                                          Technical  challenge level numeric
89                                       6 verrines                      3.0
26                                 floating islands                      NaN
82                                  torta setteveli                      2.0
112               8 pâtisserie-style layered slices                      NaN
98   3 raspberry éclairs & 3 salted caramel éclairs                      NaN
102               danish kransekake cornucopia cake                      NaN
59                                  2 herb fougasse                      NaN
114                                 red velvet cake                      2.0
23                               25 fondant fancies                      2.0
117                                     steak tacos                      NaN


In [68]:
episode_info_df_clean

,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric
0,8/17/2010,1,1,cake,cake,180.0,victoria sandwich,NaN,chocolate celebration cake,NaN,1.0
1,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,1.0
2,8/31/2010,1,3,bread,signature bread,210.0,cob,150.0,12 sweet and 12 savoury rolls,360.0,NaN
3,9/7/2010,1,4,pudding,classic pudding,150.0,mini hot lemon soufflés,40.0,"crumble, bread, & suet puddings",300.0,NaN
4,9/14/2010,1,5,pastry,savoury pie,150.0,cornish pasties,90.0,savory canapés and sweet tartlets,300.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,12 individual spiced buns,165.0,lemon and thyme drizzle cake,90.0,floral dessert,240.0,NaN
130,11/7/2023,14,7,desserts,8 creme caramels,165.0,orange and ginger treacle puddings,90.0,meringue bombe,NaN,2.0
131,11/14/2023,14,8,party,12 sausage rolls,150.0,caterpiller cake,150.0,anything but beige buffet,270.0,2.0
132,11/21/2023,14,9,patisserie,24 financiers,120.0,tart aux pommes,150.0,millefoglie,240.0,2.0


In [69]:
episode_info_df_clean.to_csv('TechnicalBakes.csv')